<a href="https://colab.research.google.com/github/stefhanieamalia-ai/ETL-E-Commerce-Orders/blob/main/ETL_Order_Product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving raw_orders.csv to raw_orders.csv
Saving raw_products.csv to raw_products.csv


In [1]:
import pandas as pd
import numpy as np

# === EXTRACT ===
# Baca data dari "sumber" (simulasi: file CSV)
orders = pd.read_csv('raw_orders.csv')
products = pd.read_csv('raw_products.csv')

# Inspeksi awal
print("=== ORDERS INFO ===")
print(f"Jumlah baris: {len(orders)}")
print(f"Kolom: {list(orders.columns)}")
print(orders.info())
print()
print("=== CEK MASALAH ===")
print(f"Duplikasi: {orders.duplicated().sum()}")
print(f"Missing values:")
print(orders.isnull().sum())
print(f"\nHarga negatif: {(orders['total_harga'] < 0).sum()}")
print(f"\nChannel unik: {orders['channel'].unique()}")
print(f"Kota unik: {orders['kota'].unique()}")

=== ORDERS INFO ===
Jumlah baris: 130
Kolom: ['order_id', 'product_id', 'product_name', 'kategori', 'quantity', 'total_harga', 'tanggal_order', 'kota', 'channel', 'status', 'customer_email']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        130 non-null    object 
 1   product_id      130 non-null    object 
 2   product_name    130 non-null    object 
 3   kategori        130 non-null    object 
 4   quantity        130 non-null    int64  
 5   total_harga     125 non-null    float64
 6   tanggal_order   130 non-null    object 
 7   kota            130 non-null    object 
 8   channel         130 non-null    object 
 9   status          130 non-null    object 
 10  customer_email  110 non-null    object 
dtypes: float64(1), int64(1), object(9)
memory usage: 11.3+ KB
None

=== CEK MASALAH ===
Duplikasi: 10
Missing values:


In [2]:
# === TRANSFORM ===

# 2a. Hapus duplikasi
print(f"Sebelum: {len(orders)} baris")
orders = orders.drop_duplicates()
print(f"Setelah hapus duplikat: {len(orders)} baris")

# 2b. Hapus baris dengan harga negatif (data error)
orders = orders[orders['total_harga'] >= 0]
print(f"Setelah hapus harga negatif: {len(orders)} baris")

# 2c. Isi missing values
orders['customer_email'] = orders['customer_email'].fillna('unknown@placeholder.com')
median_harga = orders['total_harga'].median()
orders['total_harga'] = orders['total_harga'].fillna(median_harga)
print(f"Missing values setelah fillna: {orders.isnull().sum().sum()}")

# 2d. Standarkan format tanggal
orders['tanggal_order'] = pd.to_datetime(orders['tanggal_order'], format='mixed')
print(f"Tipe tanggal: {orders['tanggal_order'].dtype}")

# 2e. Standarkan teks (lowercase lalu title case)
orders['kota'] = orders['kota'].str.strip().str.title()
orders['channel'] = orders['channel'].str.strip().str.lower().str.replace(' ', '_')
print(f"Channel setelah standarisasi: {orders['channel'].unique()}")
print(f"Kota setelah standarisasi: {orders['kota'].unique()}")

# 2f. Buat kolom baru: bulan dan kategori harga
orders['bulan'] = orders['tanggal_order'].dt.month_name()
orders['kategori_harga'] = np.where(
    orders['total_harga'] < 500000, 'kecil',
    np.where(orders['total_harga'] <= 2000000, 'sedang', 'besar')
)

print(f"\nDistribusi kategori harga:")
print(orders['kategori_harga'].value_counts())

Sebelum: 130 baris
Setelah hapus duplikat: 120 baris
Setelah hapus harga negatif: 110 baris
Missing values setelah fillna: 0
Tipe tanggal: datetime64[ns]
Channel setelah standarisasi: ['website' 'mobile_app' 'marketplace']
Kota setelah standarisasi: ['Solo' 'Bandung' 'Makassar' 'Malang' 'Surabaya' 'Yogyakarta' 'Medan'
 'Semarang' 'Denpasar' 'Jakarta']

Distribusi kategori harga:
kategori_harga
sedang    51
besar     49
kecil     10
Name: count, dtype: int64


In [3]:
# === VALIDATE ===
print("=== VALIDASI DATA BERSIH ===")

checks = {
    'Tidak ada duplikat': orders.duplicated().sum() == 0,
    'Tidak ada missing value': orders.isnull().sum().sum() == 0,
    'Tidak ada harga negatif': (orders['total_harga'] < 0).sum() == 0,
    'Tanggal tipe datetime': str(orders['tanggal_order'].dtype) == 'datetime64[ns]',
    'Channel konsisten': len(orders['channel'].unique()) <= 3,
}

for check, passed in checks.items():
    status = '✅' if passed else '❌'
    print(f"  {status} {check}")

all_passed = all(checks.values())
print(f"\nHasil: {'SEMUA LOLOS' if all_passed else 'ADA YANG GAGAL'}")

=== VALIDASI DATA BERSIH ===
  ✅ Tidak ada duplikat
  ✅ Tidak ada missing value
  ✅ Tidak ada harga negatif
  ✅ Tanggal tipe datetime
  ✅ Channel konsisten

Hasil: SEMUA LOLOS


In [4]:
# === LOAD ===
# Di dunia nyata: load ke BigQuery / data warehouse
# Di exercise ini: simpan ke CSV bersih

orders_clean = orders[[
    'order_id', 'product_id', 'product_name', 'kategori',
    'quantity', 'total_harga', 'tanggal_order', 'kota',
    'channel', 'status', 'customer_email',
    'bulan', 'kategori_harga'
]]

orders_clean.to_csv('orders_clean.csv', index=False)
print(f"Data bersih disimpan: orders_clean.csv ({len(orders_clean)} baris)")

# Buat summary report
summary = orders_clean.groupby('kategori').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_harga', 'sum'),
    avg_revenue=('total_harga', 'mean')
).round(0)

print("\n=== SUMMARY PER KATEGORI ===")
print(summary)

summary.to_csv('summary_report.csv')
print("\nSummary disimpan: summary_report.csv")

Data bersih disimpan: orders_clean.csv (110 baris)

=== SUMMARY PER KATEGORI ===
            total_orders  total_revenue  avg_revenue
kategori                                            
Elektronik            81    435180000.0    5372593.0
Furniture             29    127350000.0    4391379.0

Summary disimpan: summary_report.csv


In [5]:
"""
Mini ETL Orchestrator
Simulasi cara kerja Apache Airflow - bisa langsung dijalankan!
Konsep: task dependency, retry, logging, validation gate
"""
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime

# === CONFIG ===
INPUT_FILE = 'raw_orders.csv'
OUTPUT_FILE = 'orders_clean.csv'
REPORT_FILE = 'summary_report.csv'
LOG_FILE = 'pipeline_log.txt'
MAX_RETRIES = 3

# === LOGGER ===
def log(task_name, status, message=""):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{timestamp}] [{task_name}] [{status}] {message}"
    print(entry)
    with open(LOG_FILE, 'a') as f:
        f.write(entry + '\n')

# === TASK RUNNER (simulates Airflow's task execution) ===
def run_task(task_name, task_func, retries=MAX_RETRIES):
    """
    Jalankan sebuah task dengan retry logic.
    Sama seperti Airflow: jika gagal, coba ulang sampai max retries.
    """
    for attempt in range(1, retries + 1):
        try:
            log(task_name, "RUNNING", f"Attempt {attempt}/{retries}")
            start = time.time()
            result = task_func()
            duration = round(time.time() - start, 2)
            log(task_name, "SUCCESS", f"Selesai dalam {duration}s")
            return result
        except Exception as e:
            log(task_name, "FAILED", f"Error: {e}")
            if attempt < retries:
                wait = attempt * 2  # exponential backoff
                log(task_name, "RETRY", f"Menunggu {wait}s sebelum retry...")
                time.sleep(wait)
            else:
                log(task_name, "FATAL", f"Gagal setelah {retries} percobaan!")
                raise

# ============================================
# TASK DEFINITIONS (setiap fungsi = 1 task)
# ============================================

def task_extract():
    """Task 1: Baca data mentah dari sumber"""
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(f"{INPUT_FILE} tidak ditemukan!")
    df = pd.read_csv(INPUT_FILE)
    log("extract", "INFO", f"Loaded {len(df)} baris dari {INPUT_FILE}")
    return df

def task_transform(df):
    """Task 2: Bersihkan dan transformasi data"""
    before = len(df)
    # Hapus duplikat
    df = df.drop_duplicates()
    log("transform", "INFO", f"Hapus {before - len(df)} duplikat")
    # Hapus harga negatif
    neg_count = (df['total_harga'] < 0).sum()
    df = df[df['total_harga'] >= 0]
    log("transform", "INFO", f"Hapus {neg_count} harga negatif")
    # Isi missing values
    df['customer_email'] = df['customer_email'].fillna('unknown@placeholder.com')
    df['total_harga'] = df['total_harga'].fillna(df['total_harga'].median())
    # Standarkan format
    df['tanggal_order'] = pd.to_datetime(df['tanggal_order'], format='mixed')
    df['kota'] = df['kota'].str.strip().str.title()
    df['channel'] = df['channel'].str.strip().str.lower().str.replace(' ', '_')
    # Kolom baru
    df['bulan'] = df['tanggal_order'].dt.month_name()
    df['kategori_harga'] = np.where(
        df['total_harga'] < 500000, 'kecil',
        np.where(df['total_harga'] <= 2000000, 'sedang', 'besar')
    )
    log("transform", "INFO", f"Output: {len(df)} baris bersih")
    return df

def task_validate(df):
    """Task 3: Validasi kualitas data - GATE sebelum load"""
    checks = {
        'zero_duplicates': df.duplicated().sum() == 0,
        'zero_nulls': df.isnull().sum().sum() == 0,
        'zero_negative_price': (df['total_harga'] < 0).sum() == 0,
        'datetime_type': str(df['tanggal_order'].dtype).startswith('datetime'),
    }
    failed = [k for k, v in checks.items() if not v]
    if failed:
        raise ValueError(f"VALIDASI GAGAL: {failed}")
    log("validate", "INFO", f"Semua {len(checks)} check PASSED ✅")
    return df

def task_load(df):
    """Task 4: Simpan ke 'warehouse' (simulasi: CSV file)"""
    df.to_csv(OUTPUT_FILE, index=False)
    log("load", "INFO", f"Data disimpan ke {OUTPUT_FILE} ({len(df)} baris)")
    return df

def task_report(df):
    """Task 5: Generate summary report"""
    summary = df.groupby('kategori').agg(
        total_orders=('order_id', 'count'),
        total_revenue=('total_harga', 'sum'),
        avg_revenue=('total_harga', 'mean')
    ).round(0)
    summary.to_csv(REPORT_FILE)
    log("report", "INFO", f"Report disimpan ke {REPORT_FILE}")
    return summary

def task_notify(summary):
    """Task 6: Kirim notifikasi (simulasi: print)"""
    total_orders = summary['total_orders'].sum()
    total_revenue = summary['total_revenue'].sum()
    message = (
        f"\n{'='*50}\n"
        f"📧 PIPELINE NOTIFICATION\n"
        f"{'='*50}\n"
        f"Status: ✅ SUCCESS\n"
        f"Waktu: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Total orders diproses: {int(total_orders)}\n"
        f"Total revenue: Rp {int(total_revenue):,}\n"
        f"Output: {OUTPUT_FILE}, {REPORT_FILE}\n"
        f"{'='*50}"
    )
    print(message)
    log("notify", "INFO", "Notifikasi terkirim")

# ============================================
# DAG DEFINITION - urutan eksekusi task
# ============================================

def run_pipeline():
    """
    DAG: etl_ecommerce_daily
    Schedule: manual (di Airflow: '0 6 * * *' = tiap hari jam 6)

    Task Flow:
      extract >> transform >> validate >> load >> report >> notify

    Jika validate gagal, pipeline BERHENTI - data kotor tidak di-load.
    """
    print("\n" + "="*50)
    print("🚀 STARTING ETL PIPELINE")
    print("="*50 + "\n")

    # Clear log
    open(LOG_FILE, 'w').close()

    start_time = time.time()

    try:
        # Task 1: Extract
        df = run_task("extract", task_extract)

        # Task 2: Transform (depends on extract)
        df = run_task("transform", lambda: task_transform(df))

        # Task 3: Validate (depends on transform)
        # INI ADALAH GATE - jika gagal, load TIDAK dijalankan
        df = run_task("validate", lambda: task_validate(df))

        # Task 4: Load (depends on validate)
        df = run_task("load", lambda: task_load(df))

        # Task 5: Report (depends on load)
        summary = run_task("report", lambda: task_report(df))

        # Task 6: Notify (depends on report)
        run_task("notify", lambda: task_notify(summary))

        total_time = round(time.time() - start_time, 2)
        log("pipeline", "COMPLETED", f"Total waktu: {total_time}s")

    except Exception as e:
        log("pipeline", "FAILED", f"Pipeline gagal: {e}")
        print(f"\n❌ PIPELINE GAGAL: {e}")
        print(f"Lihat {LOG_FILE} untuk detail.")

if __name__ == '__main__':
    run_pipeline()


🚀 STARTING ETL PIPELINE

[2026-07-17 06:16:36] [extract] [RUNNING] Attempt 1/3
[2026-07-17 06:16:36] [extract] [INFO] Loaded 130 baris dari raw_orders.csv
[2026-07-17 06:16:36] [extract] [SUCCESS] Selesai dalam 0.0s
[2026-07-17 06:16:36] [transform] [RUNNING] Attempt 1/3
[2026-07-17 06:16:36] [transform] [INFO] Hapus 10 duplikat
[2026-07-17 06:16:36] [transform] [INFO] Hapus 5 harga negatif
[2026-07-17 06:16:36] [transform] [INFO] Output: 110 baris bersih
[2026-07-17 06:16:36] [transform] [SUCCESS] Selesai dalam 0.01s
[2026-07-17 06:16:36] [validate] [RUNNING] Attempt 1/3
[2026-07-17 06:16:36] [validate] [INFO] Semua 4 check PASSED ✅
[2026-07-17 06:16:36] [validate] [SUCCESS] Selesai dalam 0.0s
[2026-07-17 06:16:36] [load] [RUNNING] Attempt 1/3
[2026-07-17 06:16:36] [load] [INFO] Data disimpan ke orders_clean.csv (110 baris)
[2026-07-17 06:16:36] [load] [SUCCESS] Selesai dalam 0.0s
[2026-07-17 06:16:36] [report] [RUNNING] Attempt 1/3
[2026-07-17 06:16:36] [report] [INFO] Report disimp

In [7]:
python etl_pipeline.py

SyntaxError: invalid syntax (3891068845.py, line 1)